# Agent 3 — Teams Alert Dispatcher

Sends Teams MessageCard alerts for failed SQL reviews.

In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
REVIEW_RESULTS = f"{CATALOG}.{SCHEMA}.uc3_review_results_b"

TEAMS_WEBHOOK_URL = dbutils.secrets.get(
    scope="agent1-secrets",
    key  ="teams-webhook-url"
)

ALERT_CHANNEL = "platform-support"

STATUS_EMOJI = {
    "CRITICAL": "🔴",
    "WARNING" : "🟡",
    "PASS"    : "🟢"
}

THEME_COLOR = {
    "CRITICAL": "FF0000",
    "WARNING" : "FFA500",
    "PASS"    : "00FF00"
}

## Load Reviews to Alert

In [0]:
reviews_to_alert = (
    spark.table(REVIEW_RESULTS)
         .filter(
             F.col("overall_status").isin(["CRITICAL", "WARNING"]) &
             (F.col("alert_sent").isNull() | (F.col("alert_sent") == False))
         )
         .orderBy(F.col("review_timestamp").desc())
         .collect()
)

reviews_to_alert = [r.asDict() for r in reviews_to_alert]

print(f"Reviews requiring alerts: {len(reviews_to_alert)}")
for r in reviews_to_alert:
    emoji = STATUS_EMOJI.get(r["overall_status"], "⚪")
    print(
        f"  {emoji} {r['target_fully_qualified_name'].split('.')[-1]:<40} "
        f"Status: {r['overall_status']}"
    )

if not reviews_to_alert:
    print("No reviews require alerts.")
    NO_REVIEWS_TO_ALERT = True
else:
    NO_REVIEWS_TO_ALERT = False

## Build and Send MessageCards

In [0]:
def build_agent3_card(review: dict) -> dict:
    """Builds a Teams MessageCard for an Agent 3 SQL review alert."""
    target     = review["target_fully_qualified_name"]
    table_name = target.split(".")[-1]
    status     = review["overall_status"]
    review_id  = review["review_id"]
    emoji      = STATUS_EMOJI.get(status, "⚪")
    color      = THEME_COLOR.get(status, "0078D4")
    detected   = datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")

    ai_summary  = review.get("ai_summary") or "No AI summary available."
    ai_action   = review.get("ai_recommended_action") or ""
    ai_category = review.get("ai_failure_category") or "N/A"
    ai_conf     = review.get("ai_confidence_score")

    check_results = []
    try:
        check_results = json.loads(review.get("check_results", "[]"))
    except Exception:
        pass

    failed_checks = [
        c for c in check_results
        if c.get("check_status") in ("CRITICAL", "WARNING")
    ]

    sections = []

    # ── Header ─────────────────────────────────────────────────────────────
    sections.append({
        "activityTitle"   : f"{emoji} {status} — SQL Review Alert",
        "activitySubtitle": "Agent 3 — Silver Transformation Review",
        "facts"           : [
            {"name": "Target Table",   "value": table_name},
            {"name": "Full Name",      "value": target},
            {"name": "Detected At",    "value": detected},
            {"name": "Overall Status", "value": f"{emoji} {status}"}
        ],
        "markdown": True
    })

    # ── Check summary ──────────────────────────────────────────────────────
    sections.append({
        "title": "📊 Review Summary",
        "facts": [
            {
                "name" : "Total Checks",
                "value": str(review.get("total_checks_run", 0))
            },
            {
                "name" : "✅ Passed",
                "value": str(review.get("total_checks_passed", 0))
            },
            {
                "name" : "🟡 Warning",
                "value": str(review.get("total_checks_warned", 0))
            },
            {
                "name" : "🔴 Critical",
                "value": str(review.get("total_checks_critical", 0))
            }
        ],
        "markdown": True
    })

    # ── Failed checks with full detail ─────────────────────────────────────
    if failed_checks:
        for check in failed_checks:
            c_emoji   = STATUS_EMOJI.get(check["check_status"], "⚪")
            c_name    = check["check_name"]
            c_level   = check["check_level"]
            findings  = check.get("findings", [])

            # Build full finding text from all findings
            finding_text = "\n".join([
                f"• {f}" for f in findings
            ])

            recommendation = check.get("recommendation", "")

            sections.append({
                "title"   : (
                    f"❌ {c_emoji} {c_name} [{c_level}]"
                ),
                "text"    : finding_text,
                "markdown": True
            })

            if recommendation and recommendation != "No action required.":
                sections.append({
                    "title"   : "Quick Fix",
                    "text"    : recommendation,
                    "markdown": True
                })

    # ── AI summary ─────────────────────────────────────────────────────────
    if ai_summary:
        sections.append({
            "title"   : "🤖 AI Code Review Summary",
            "text"    : ai_summary,
            "markdown": True
        })

        if ai_action:
            sections.append({
                "title"   : "🔧 Recommended Fix",
                "text"    : ai_action,
                "markdown": True
            })

        sections.append({
            "title": "AI Details",
            "facts": [
                {
                    "name" : "Failure Category",
                    "value": ai_category
                },
                {
                    "name" : "AI Confidence",
                    "value": f"{ai_conf:.0%}" if ai_conf else "N/A"
                }
            ],
            "markdown": True
        })

    # ── Footer ─────────────────────────────────────────────────────────────
    sections.append({
        "title": "🔍 Audit",
        "facts": [
            {
                "name" : "Review ID",
                "value": review_id
            },
            {
                "name" : "Catalog.Schema",
                "value": f"{CATALOG}.{SCHEMA}"
            }
        ],
        "markdown": True
    })

    return {
        "@type"     : "MessageCard",
        "@context"  : "http://schema.org/extensions",
        "themeColor": color,
        "summary"   : f"Agent 3 SQL Review Alert: {table_name} {status}",
        "title"     : f"{emoji} {status}: {table_name} SQL Review",
        "sections"  : sections
    }


def mark_alert_sent(review_id: str, success: bool):
    channel = ALERT_CHANNEL if success else "FAILED"
    spark.sql(f"""
        UPDATE {REVIEW_RESULTS}
        SET
            alert_sent              = {str(success).lower()},
            alert_sent_timestamp    = '{datetime.now()}',
            alert_channel           = '{channel}'
        WHERE review_id = '{review_id}'
    """)


# ── Dispatch ───────────────────────────────────────────────────────────────

alerts_sent   = 0
alerts_failed = 0

if NO_REVIEWS_TO_ALERT:
    print("Skipping dispatch — no reviews require alerts.")
else:
    for review in reviews_to_alert:
        review_id  = review["review_id"]
        table_name = review["target_fully_qualified_name"].split(".")[-1]
        status     = review["overall_status"]
        emoji      = STATUS_EMOJI.get(status, "⚪")

        print(f"\n{emoji} {table_name}  |  {status}")

        try:
            card    = build_agent3_card(review)
            resp    = requests.post(
                TEAMS_WEBHOOK_URL,
                headers={"Content-Type": "application/json"},
                data=json.dumps(card),
                timeout=30
            )
            success = resp.status_code in (200, 202)
        except Exception as e:
            print(f"  Error: {str(e)}")
            success = False

        mark_alert_sent(review_id, success)

        if success:
            alerts_sent += 1
            print(f"  Alert sent ✅")
        else:
            alerts_failed += 1
            print(f"  Alert failed ❌")

In [0]:
print("\n" + "=" * 60)
print("AGENT 3 ALERT DISPATCHER COMPLETE")
print("=" * 60)
print(f"Alerts sent   : {alerts_sent}")
print(f"Alerts failed : {alerts_failed}")
print(f"Channel       : {ALERT_CHANNEL}")
print(f"Completed at  : {datetime.now()}")
print("=" * 60)